### **BILBIOTECAS**

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np 

from feature_engine.selection import DropConstantFeatures, DropDuplicateFeatures, SmartCorrelatedSelection

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

from projeto_ml.pipeline import Pipeline_Data

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [3]:
LOCAL_DADOS = '../data/gold'

In [4]:
df = pd.read_parquet(f'{LOCAL_DADOS}/churn.parquet')

df.head(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,0


In [5]:
df_feat = df.copy()   

# 1. Tempo e Valor
df_feat['Is_New_Customer'] = (df_feat['tenure'] <= 6).astype(int)    

# 2. Engajamento
servicos = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_feat['Total_Services'] = df_feat[servicos].apply(lambda x: (x == 'Yes').sum(), axis=1)
df_feat['Has_Internet'] = (df_feat['InternetService'] != 'No').astype(int)    

# 3. Contrato e Pagamento
df_feat['Is_Monthly_Contract'] = (df_feat['Contract'] == 'Month-to-month').astype(int)
df_feat['Is_Auto_Payment'] = df_feat['PaymentMethod'].isin(['Bank transfer (automatic)', 'Credit card (automatic)']).astype(int)    

# 4. Perfil Familiar
df_feat['Family_Size'] = 1 + (df_feat['Partner'] == 'Yes').astype(int) + (df_feat['Dependents'] == 'Yes').astype(int)    

# 5. Risco Financeiro
df_feat['Charge_to_Tenure_Ratio'] = df_feat['MonthlyCharges'] / (df_feat['tenure'] + 1)    


### **Pipeline Dados**

In [6]:
X = df_feat.drop(columns=['Churn'])
y = df_feat['Churn']

pipe = Pipeline_Data(X=X, y=y, test_size=.30)

X_train, y_train, X_val, y_val, X_test, y_test, pipeline = pipe.split_data()

Shape original do treino: (4930, 26)
Shape após limpeza: (4930, 26)

Shape test: (1057, 26)

Shape val: (1056, 26)

--- Features Constantes Removidas ---
[]

--- Features Correlacionadas Removidas ---
[]

--- Pares de Features Correlacionadas (Correlação > 0.85) ---
[]

--- Features Duplicadas Removidas ---
set()

--- Grupos de Features Duplicadas ---
[]


#### **BASE_TREINAMENTO**

In [7]:
X_train['Churn'] = y_train
X_train.to_parquet(f'{LOCAL_DADOS}/train.parquet')

#### **BASE_VALIDACAO**

In [8]:
X_test['Churn'] = y_test
X_test.to_parquet(f'{LOCAL_DADOS}/test.parquet')

#### **BASE_TESTE***

In [9]:
X_val['Churn'] = y_val
X_val.to_parquet(f'{LOCAL_DADOS}/val.parquet')